# Setup

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, confusion_matrix



In [6]:
pip install mlflow dagshub

Note: you may need to restart the kernel to use updated packages.


In [7]:
import mlflow
import mlflow.sklearn
import dagshub

dagshub.init(repo_owner='adzid23', repo_name='Freeuni_ML_Fraud_Detection', mlflow=True)
mlflow.set_experiment("LR_Training")

Initialized MLflow to track repo "adzid23/Freeuni_ML_Fraud_Detection"

Repository adzid23/Freeuni_ML_Fraud_Detection initialized!

2026/05/03 17:46:46 INFO mlflow.tracking.fluent: Experiment with name 'LR_Training' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/55d05d0c0c8142b1aa29832f0ce2af7d', creation_time=1777830406604, experiment_id='1', last_update_time=1777830406604, lifecycle_stage='active', name='LR_Training', tags={}, trace_location=None, workspace='default'>

# Data Loading

In [8]:
DATA_PATH = '/kaggle/input/competitions/ieee-fraud-detection'

train_transaction = pd.read_csv(f'{DATA_PATH}/train_transaction.csv')
train_identity    = pd.read_csv(f'{DATA_PATH}/train_identity.csv')

df = train_transaction.merge(train_identity, on='TransactionID', how='left')

print(f"Shape after merge: {df.shape}")
print(f"Fraud rate: {df['isFraud'].mean():.4f}")

Shape after merge: (590540, 434)
Fraud rate: 0.0350


# Cleaning

In [9]:
with mlflow.start_run(run_name="LR_Cleaning"):
    
    y = df['isFraud']
    X = df.drop(columns=['isFraud', 'TransactionID'])
    
    null_frac = X.isnull().mean()
    cols_to_drop = null_frac[null_frac > 0.5].index.tolist()
    X = X.drop(columns=cols_to_drop)
    
    X = X.drop(columns=['TransactionDT'], errors='ignore')
    
    print(f"Dropped {len(cols_to_drop)} high-null columns")
    print(f"Shape after cleaning: {X.shape}")
    
    mlflow.log_params({
        "null_threshold": 0.5,
        "dropped_cols": len(cols_to_drop),
    })
    mlflow.log_metric("cols_after_cleaning", X.shape[1])

Dropped 214 high-null columns
Shape after cleaning: (590540, 217)
🏃 View run LR_Cleaning at: https://dagshub.com/adzid23/Freeuni_ML_Fraud_Detection.mlflow/#/experiments/1/runs/09120cfac74644bcb929f6d0bfe9f9bd
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Fraud_Detection.mlflow/#/experiments/1


# Feature Engineering

In [10]:
with mlflow.start_run(run_name="LR_Feature_Engineering"):
    
    X['hour'] = (df['TransactionDT'] // 3600) % 24
    X['day']  = (df['TransactionDT'] // (3600 * 24)) % 7
    X['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
    
    cat_cols = X.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        freq_map = X[col].value_counts(normalize=True).to_dict()
        X[col] = X[col].map(freq_map).fillna(0)
    
    print(f"Encoded {len(cat_cols)} categorical columns via frequency encoding")
    print(f"Shape after feature engineering: {X.shape}")
    
    mlflow.log_params({
        "new_features": "hour, day, TransactionAmt_log",
        "cat_encoding": "frequency",
        "cat_cols_encoded": len(cat_cols)
    })
    mlflow.log_metric("total_features", X.shape[1])

Encoded 9 categorical columns via frequency encoding
Shape after feature engineering: (590540, 220)
🏃 View run LR_Feature_Engineering at: https://dagshub.com/adzid23/Freeuni_ML_Fraud_Detection.mlflow/#/experiments/1/runs/2e0d26d4ce4a4cbca026b8195f4d93bd
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Fraud_Detection.mlflow/#/experiments/1


# Feature Selection

In [11]:
with mlflow.start_run(run_name="LR_Feature_Selection"):
    
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    low_var_cols = [c for c in num_cols if X[c].std() < 0.01]
    X = X.drop(columns=low_var_cols)
    print(f"Dropped {len(low_var_cols)} near-constant columns")
    
    corr_matrix = X.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_cols = [c for c in upper.columns if any(upper[c] > 0.9)]
    X = X.drop(columns=high_corr_cols)
    print(f"Dropped {len(high_corr_cols)} highly correlated columns")
    
    print(f"Final feature count: {X.shape[1]}")
    
    mlflow.log_params({
        "variance_threshold": 0.01,
        "correlation_threshold": 0.9,
    })
    mlflow.log_metric("dropped_low_var", len(low_var_cols))
    mlflow.log_metric("dropped_high_corr", len(high_corr_cols))
    mlflow.log_metric("features_selected", X.shape[1])

Dropped 2 near-constant columns
Dropped 102 highly correlated columns
Final feature count: 116
🏃 View run LR_Feature_Selection at: https://dagshub.com/adzid23/Freeuni_ML_Fraud_Detection.mlflow/#/experiments/1/runs/74c06e13223b47a7b9cf394307cbc787
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Fraud_Detection.mlflow/#/experiments/1


# Training

## Run 1 - Baseline

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

with mlflow.start_run(run_name="LR_Baseline"):
    
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   LogisticRegression(
            solver='lbfgs',
            max_iter=1000,
            class_weight='balanced',
            n_jobs=-1,
            random_state=42
        ))
    ])
    
    pipeline.fit(X_train, y_train)
    
    train_auc = roc_auc_score(y_train, pipeline.predict_proba(X_train)[:, 1])
    test_auc  = roc_auc_score(y_test,  pipeline.predict_proba(X_test)[:, 1])
    gap       = train_auc - test_auc
    
    print(f"Train AUC: {train_auc:.4f}")
    print(f"Test AUC:  {test_auc:.4f}")
    print(f"Gap:       {gap:.4f}")
    
    mlflow.log_params({
        "solver": "lbfgs",
        "max_iter": 1000,
        "class_weight": "balanced",
        "run": "baseline"
    })
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("test_auc",  test_auc)
    mlflow.log_metric("overfit_gap", gap)

Train: (472432, 116) | Test: (118108, 116)
Train AUC: 0.8243
Test AUC:  0.8215
Gap:       0.0028
🏃 View run LR_Baseline at: https://dagshub.com/adzid23/Freeuni_ML_Fraud_Detection.mlflow/#/experiments/1/runs/67453e7d84a34250abded6a24cef71d9
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Fraud_Detection.mlflow/#/experiments/1


## Run 2 - Regularization Sweep (C values)

In [13]:
with mlflow.start_run(run_name="LR_Regularization_Sweep"):
    
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    
    for C in [0.001, 0.01, 0.1, 1.0, 10.0]:
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',  StandardScaler()),
            ('model',   LogisticRegression(
                C=C,
                solver='saga',  
                max_iter=300,
                tol=1e-3,
                class_weight='balanced',
                n_jobs=-1,
                random_state=42
            ))
        ])
        
        scores = cross_val_score(pipe, X_train, y_train,
                                 cv=skf, scoring='roc_auc', n_jobs=-1)
        
        mlflow.log_metric("cv_auc", scores.mean(), step=int(C * 1000))
        print(f"C={C:.3f} → CV AUC: {scores.mean():.4f} ± {scores.std():.4f}")

C=0.001 → CV AUC: 0.8215 ± 0.0009


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


C=0.010 → CV AUC: 0.8220 ± 0.0008


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


C=0.100 → CV AUC: 0.8220 ± 0.0008


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


C=1.000 → CV AUC: 0.8220 ± 0.0008


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


C=10.000 → CV AUC: 0.8220 ± 0.0008
🏃 View run LR_Regularization_Sweep at: https://dagshub.com/adzid23/Freeuni_ML_Fraud_Detection.mlflow/#/experiments/1/runs/57c8f2b3d57e45c882b0fd2742d12964
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Fraud_Detection.mlflow/#/experiments/1


## Run 3 - L1 vs L2 penalty + fixing convergence

In [14]:
with mlflow.start_run(run_name="LR_L1_vs_L2"):
    
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    
    for penalty in ['l1', 'l2']:
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',  StandardScaler()),
            ('model',   LogisticRegression(
                C=1.0,
                penalty=penalty,
                solver='saga',
                max_iter=1000, 
                tol=1e-3,
                class_weight='balanced',
                n_jobs=-1,
                random_state=42
            ))
        ])
        
        scores = cross_val_score(pipe, X_train, y_train,
                                 cv=skf, scoring='roc_auc', n_jobs=-1)
        
        mlflow.log_metric(f"{penalty}_cv_auc", scores.mean())
        print(f"penalty={penalty} → CV AUC: {scores.mean():.4f} ± {scores.std():.4f}")

penalty=l1 → CV AUC: 0.8221 ± 0.0008
penalty=l2 → CV AUC: 0.8221 ± 0.0008
🏃 View run LR_L1_vs_L2 at: https://dagshub.com/adzid23/Freeuni_ML_Fraud_Detection.mlflow/#/experiments/1/runs/e8e7fec2927b445f9dfbe2c0e3f85ef2
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Fraud_Detection.mlflow/#/experiments/1


## Final Model - Save to Model Registry

In [15]:
with mlflow.start_run(run_name="LR_FinalModel"):
    
    final_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   LogisticRegression(
            C=1.0,
            penalty='l2',
            solver='saga',
            max_iter=1000,
            tol=1e-2,
            class_weight='balanced',
            n_jobs=-1,
            random_state=42
        ))
    ])
    
    final_pipeline.fit(X_train, y_train)
    
    train_auc = roc_auc_score(y_train, final_pipeline.predict_proba(X_train)[:, 1])
    test_auc  = roc_auc_score(y_test,  final_pipeline.predict_proba(X_test)[:, 1])
    
    print(f"Train AUC: {train_auc:.4f}")
    print(f"Test AUC:  {test_auc:.4f}")
    
    mlflow.log_params({
        "C": 1.0,
        "penalty": "l2",
        "solver": "saga",
        "max_iter": 1000,
        "tol": 1e-2,
        "class_weight": "balanced"
    })
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("test_auc",  test_auc)
    
    mlflow.sklearn.log_model(
        final_pipeline,
        artifact_path="lr_fraud_pipeline",
        registered_model_name="LR_Fraud_Pipeline"
    )
    print("Model saved to Model Registry as LR_Fraud_Pipeline")

Train AUC: 0.8220
Test AUC:  0.8204


2026/05/03 19:35:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 19:35:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'LR_Fraud_Pipeline'.
2026/05/03 19:35:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LR_Fraud_Pipeline, version 1
Created version '1' of model 'LR_Fraud_Pipeline'.


Model saved to Model Registry as LR_Fraud_Pipeline
🏃 View run LR_FinalModel at: https://dagshub.com/adzid23/Freeuni_ML_Fraud_Detection.mlflow/#/experiments/1/runs/38598045d2ff4bfbb72cb57b16a6cec4
🧪 View experiment at: https://dagshub.com/adzid23/Freeuni_ML_Fraud_Detection.mlflow/#/experiments/1
